# 🚀 TabDrift: Standardized Latents + EMA Model Averaging

- **Normalization**: Per-dimension standard deviation scaling `(z - mean) / std` (preserves unit-variance hypersphere for multi-scale kernels)
- **Stabilization**: Exponential Moving Average (EMA, decay=0.999) model weights auto-loaded during sampling
- **Execution**: 4000 epochs, 10.6M parameter Residual MLP Generator, drift scale $c=1.5$

In [ ]:
# 1. Clone Private GitHub Repository
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

GITHUB_USER = "ahmed-fouad-lagha"
REPO_NAME = "tabsyn"

!git clone https://{github_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd {REPO_NAME}

In [ ]:
# 2. Install Dependencies
!pip install -q icecream zero tomli tomli-w category_encoders executing asttokens prdc catboost lightgbm

In [ ]:
# 3. Verify GPU
!nvidia-smi

In [ ]:
# 4. Train TabDrift (Standardized Latents + EMA)
!PYTHONPATH=. python tabsyn/drift_train.py \
    --dataname adult \
    --gpu 0 \
    --epochs 4000 \
    --batch_size 4096 \
    --lr 1e-4 \
    --hidden_size 1024 \
    --temperatures 0.1 0.5 1.0 2.0 \
    --drift_scale 1.5 \
    --patience 4000

In [ ]:
# 5. Generate Synthetic Data (1-Step Direct Pass using EMA Weights)
!PYTHONPATH=. python tabsyn/drift_sample.py \
    --dataname adult \
    --gpu 0 \
    --steps 1

In [ ]:
# 6. Full Multi-Classifier Machine Learning Efficacy Evaluation
!python eval/eval_mle.py --dataname adult --model tabdrift

In [ ]:
# 7. Print Benchmark JSON Results
import json
with open('eval/mle/adult/tabdrift.json', 'r') as f:
    scores = json.load(f)
print(json.dumps(scores, indent=4))